In [1]:
import sys
import os
import datasets
from datasets import Dataset, concatenate_datasets
from typing import List, Dict, Any
import random
import json

from transformers import AutoTokenizer

# Add the project root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.utils.dataset_tokenization import extract_deleted_text
from src.utils.formatting import apply_del_w_tokens

In [24]:
def format_single_correct_sample(sample: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Formats a single raw sample into one or more training samples."""

    sample_kwargs = {
        "question": sample["question"],
        "answer": str(sample.get("answer", [""])[0]),
    }
    if "is_answerable" in sample:
        sample_kwargs["is_answerable"] = sample["is_answerable"]
    else:
        sample_kwargs["context"] = sample.get("context", "")
    
    formatted_samples = []
    responses = sample["responses"]
    # sort responses by length
    responses.sort(key=len)
    # samples_to_include = 1 if sample["is_answerable"] else 2
    samples_to_include = 1


    for i in range(samples_to_include):
        formatted_samples.append({
            "input": sample["input"],
            "incorrect_response": "",
            "errors": [],
            "hallucinated_text": [],
            "correct_response": responses[i],
            "additional_info": sample_kwargs,
        })

    return formatted_samples

In [3]:
def format_single_incorrect_sample(sample: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Formats a single raw sample into one or more training samples."""
    if sample.get("wrong_response_number", 0) == 0:
        return []

    for correct_response in sample["corrected_responses"]:
        if ("<DEL_S>" not in correct_response and "<DEL_A>" not in correct_response):
            return []

    sample_kwargs = {
        "question": sample["question"],
        "answer": str(sample.get("answer", [""])[0]),
    }
    if "is_answerable" in sample:
        sample_kwargs["is_answerable"] = sample["is_answerable"]
    else:
        sample_kwargs["context"] = sample.get("context", "")
    
    formatted_samples = []
    correct_response_index = 0

    # samples_to_include = 2 if random.random() < 0.35 else 3
    samples_to_include = 5

    for i, is_verified in enumerate(sample.get("verified_response_mask", [])):
        if is_verified:
            hallucinated_text = []
            for error in sample["errors_to_correct"][i]:
                deleted_text = extract_deleted_text(error["correction"])
                if deleted_text:
                    hallucinated_text.append(deleted_text)

            if "<DEL_W>" in sample["corrected_responses"][correct_response_index]:
                sample["corrected_responses"][correct_response_index] = apply_del_w_tokens(sample["corrected_responses"][correct_response_index])

            if hallucinated_text:
                formatted_samples.append({
                    "input": sample["input"],
                    "incorrect_response": sample["responses_to_correct"][i],
                    "errors": sample["errors_to_correct"][i],
                    "hallucinated_text": hallucinated_text,
                    "correct_response": sample["corrected_responses"][correct_response_index],
                    "additional_info": sample_kwargs,
                })
                correct_response_index += 1
        
        if correct_response_index >= samples_to_include:
            break

    return formatted_samples

In [4]:
context_data_path = "../../dataset/processed_data/train/rajpurkar_squad_processed.json"
context_dataset = datasets.load_dataset("json", data_files=context_data_path)

math_data_path = "../../dataset/processed_data/train/UMWP_processed.json"
math_dataset = datasets.load_dataset("json", data_files=math_data_path)


context_corrected_data_path = "../../dataset/processed_data/train/s2_context_source.json"
context_corrected_dataset = datasets.load_dataset("json", data_files=context_corrected_data_path)

math_corrected_data_path = "../../dataset/processed_data/train/s2_math_source.json"
math_corrected_dataset = datasets.load_dataset("json", data_files=math_corrected_data_path)

In [5]:
print(context_dataset)
print(math_dataset)
print(math_corrected_dataset)
print(context_corrected_dataset)

DatasetDict({
    train: Dataset({
        features: ['input', 'question', 'context', 'answer', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'errors_to_correct', 'corrected_responses', 'verified_response_mask'],
        num_rows: 87557
    })
})
DatasetDict({
    train: Dataset({
        features: ['input', 'question', 'answer', 'is_answerable', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'errors_to_correct', 'corrected_responses', 'verified_response_mask'],
        num_rows: 4412
    })
})
DatasetDict({
    train: Dataset({
        features: ['input', 'question', 'answer', 'is_answerable', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'errors_to_correct', 'corrected_responses', 'verified_response_mask'],
        num_rows: 166
    })
})
DatasetDict({
    train: Dataset({
        features: ['input', 'question', 'context', 'answer', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'err

## Math QA dataset prep

Take:
- 1k of incorrect answerable samples
- 5k of incorrect unanswerable samples
- 4k of correct samples

In [6]:
final_math_dataset = []

### Find incorrect samples that are answerable

In [7]:
incorrect_answerable_samples = math_corrected_dataset.filter(lambda x: x["wrong_response_number"] > 0 and x["is_answerable"])
incorrect_answerable_samples = incorrect_answerable_samples["train"]
print(incorrect_answerable_samples)

Dataset({
    features: ['input', 'question', 'answer', 'is_answerable', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'errors_to_correct', 'corrected_responses', 'verified_response_mask'],
    num_rows: 131
})


In [8]:
incorrect_answerable_samples[2]

{'input': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a meticulous AI mathematician. Your task is to solve the following math problem.\n\nFollow these steps carefully:\n1. **Analyze the problem:** First, understand the given information and what is being asked.\n2. **Assess solvability:** Determine if the problem is solvable. A problem might be unsolvable if it's illogical, contains contradictions, or lacks necessary information.\n3. **Solve or Explain:**\n   - **If solvable:** Provide a step-by-step solution, showing all your reasoning and calculations, and then clearly state the final numerical answer.\n   - **If unsolvable:** State that the problem cannot be answered and provide a concise explanation.\n\nYour entire response should only contain the solution and final answer (or the explanation for unsolvable problems). Do not add any conversational headers or extraneous text.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhile Greg was camping with

In [9]:
for sample in incorrect_answerable_samples:
    final_math_dataset.extend(format_single_incorrect_sample(sample))

random.shuffle(final_math_dataset)
print(len(final_math_dataset))

191


In [10]:
final_math_dataset[0]

{'input': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a meticulous AI mathematician. Your task is to solve the following math problem.\n\nFollow these steps carefully:\n1. **Analyze the problem:** First, understand the given information and what is being asked.\n2. **Assess solvability:** Determine if the problem is solvable. A problem might be unsolvable if it's illogical, contains contradictions, or lacks necessary information.\n3. **Solve or Explain:**\n   - **If solvable:** Provide a step-by-step solution, showing all your reasoning and calculations, and then clearly state the final numerical answer.\n   - **If unsolvable:** State that the problem cannot be answered and provide a concise explanation.\n\nYour entire response should only contain the solution and final answer (or the explanation for unsolvable problems). Do not add any conversational headers or extraneous text.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nDanielle wants to make her 

### Find incorrect samples that are unanswerable

In [11]:
incorrect_unanswerable_samples = math_corrected_dataset.filter(lambda x: x["wrong_response_number"] > 0 and not x["is_answerable"])
incorrect_unanswerable_samples = incorrect_unanswerable_samples["train"]
print(incorrect_unanswerable_samples)

Dataset({
    features: ['input', 'question', 'answer', 'is_answerable', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'errors_to_correct', 'corrected_responses', 'verified_response_mask'],
    num_rows: 35
})


In [12]:
incorrect_unanswerable_samples[0]

{'input': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a meticulous AI mathematician. Your task is to solve the following math problem.\n\nFollow these steps carefully:\n1. **Analyze the problem:** First, understand the given information and what is being asked.\n2. **Assess solvability:** Determine if the problem is solvable. A problem might be unsolvable if it's illogical, contains contradictions, or lacks necessary information.\n3. **Solve or Explain:**\n   - **If solvable:** Provide a step-by-step solution, showing all your reasoning and calculations, and then clearly state the final numerical answer.\n   - **If unsolvable:** State that the problem cannot be answered and provide a concise explanation.\n\nYour entire response should only contain the solution and final answer (or the explanation for unsolvable problems). Do not add any conversational headers or extraneous text.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nSasha made 30 chocolate muf

In [13]:
# for i in range(2000):
#     final_math_dataset.extend(format_single_incorrect_sample(incorrect_unanswerable_samples[i]))

for sample in incorrect_unanswerable_samples:
    final_math_dataset.extend(format_single_incorrect_sample(sample))

random.shuffle(final_math_dataset)
print(len(final_math_dataset))

291


### Find correct samples

In [14]:
# find data samples for math dataset with no errors 
correct_samples = math_dataset.filter(lambda x: x["wrong_response_number"] == 0)
correct_samples = correct_samples["train"]
print(correct_samples)

Dataset({
    features: ['input', 'question', 'answer', 'is_answerable', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'errors_to_correct', 'corrected_responses', 'verified_response_mask'],
    num_rows: 1575
})


In [15]:
# for sample in correct_samples:
#     final_math_dataset.extend(format_single_correct_sample(sample))
# final_math_dataset = []
for i in range(410):
    final_math_dataset.extend(format_single_correct_sample(correct_samples[i]))

random.shuffle(final_math_dataset)
print(len(final_math_dataset))

731


In [16]:
final_math_dataset[0]

{'input': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a meticulous AI mathematician. Your task is to solve the following math problem.\n\nFollow these steps carefully:\n1. **Analyze the problem:** First, understand the given information and what is being asked.\n2. **Assess solvability:** Determine if the problem is solvable. A problem might be unsolvable if it's illogical, contains contradictions, or lacks necessary information.\n3. **Solve or Explain:**\n   - **If solvable:** Provide a step-by-step solution, showing all your reasoning and calculations, and then clearly state the final numerical answer.\n   - **If unsolvable:** State that the problem cannot be answered and provide a concise explanation.\n\nYour entire response should only contain the solution and final answer (or the explanation for unsolvable problems). Do not add any conversational headers or extraneous text.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nMax watches a show every da

In [17]:
output_path = os.path.join(project_root, "math_qa_train_dataset_s2.json")
with open(output_path, "w") as f:
    json.dump(final_math_dataset, f, indent=4)

## Context QA dataset

Take:
- 16k of incorrect samples
- 8k of correct samples

In [19]:
final_context_dataset = []

### Find incorrect samples

In [20]:
# find data samples for context dataset with no errors 
incorrect_samples = context_corrected_dataset.filter(lambda x: x["wrong_response_number"] > 0)
incorrect_samples = incorrect_samples["train"]
print(incorrect_samples)

Filter:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['input', 'question', 'context', 'answer', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'errors_to_correct', 'corrected_responses', 'verified_response_mask'],
    num_rows: 100
})


In [21]:
def format_single_incorrect_sample(sample: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Formats a single raw sample into one or more training samples."""
    if sample.get("wrong_response_number", 0) == 0:
        return []

    for correct_response in sample["corrected_responses"]:
        if "<DEL_W>" in correct_response:
            return []

    answer = sample["answer"][0].lower()
    counter = 0
    for response in sample["responses_to_correct"]:
        if answer in response.lower():
            counter += 1
        if counter > 2:
            return []

    sample_kwargs = {
        "question": sample["question"],
        "answer": str(sample.get("answer", [""])[0]),
    }
    if "is_answerable" in sample:
        sample_kwargs["is_answerable"] = sample["is_answerable"]
    else:
        sample_kwargs["context"] = sample.get("context", "")
    
    formatted_samples = []
    correct_response_index = 0

    # samples_to_include = 1 if random.random() < 0.05 else 2
    samples_to_include = 5

    for i, is_verified in enumerate(sample.get("verified_response_mask", [])):
        if is_verified:
            hallucinated_text = []
            for error in sample["errors_to_correct"][i]:
                deleted_text = extract_deleted_text(error["correction"])
                if deleted_text:
                    hallucinated_text.append(deleted_text)

            if hallucinated_text:
                formatted_samples.append({
                    "input": sample["input"],
                    "incorrect_response": sample["responses_to_correct"][i],
                    "errors": sample["errors_to_correct"][i],
                    "hallucinated_text": hallucinated_text,
                    "correct_response": sample["corrected_responses"][correct_response_index],
                    "additional_info": sample_kwargs,
                })
                correct_response_index += 1
        
        if correct_response_index >= samples_to_include:
            break

    return formatted_samples

In [22]:
final_context_dataset = []
samples_to_include = 100
for i in range(samples_to_include):
    final_context_dataset.extend(format_single_incorrect_sample(incorrect_samples[i]))

random.shuffle(final_context_dataset)
print(len(final_context_dataset))

208


In [23]:
final_context_dataset[0]

{'input': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a specialized question-answering AI. Your task is to give a concise answer to the question using *only* the provided context. Make sure to always give an answer.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nContext:\n'''\nThe University of Plymouth enrolls 25,895 total students as of 2014/15 (22nd largest in the UK out of 165). It also employs 3,000 staff with an annual income of around £160 million. It was founded in 1992 from Polytechnic South West (formerly Plymouth Polytechnic) following the Further and Higher Education Act 1992. It has courses in maritime business, marine engineering, marine biology and Earth, ocean and environmental sciences, surf science, shipping and logistics. The university formed a joint venture with the fellow Devonian University of Exeter in 2000, establishing the Peninsula College of Medicine and Dentistry. The college is ranked 8th out of 30 universities in the UK i

### Find correct samples

In [25]:
# find data samples for context dataset with no errors 
correct_samples = context_dataset.filter(lambda x: x["wrong_response_number"] == 0)
correct_samples = correct_samples["train"]
print(correct_samples)

Dataset({
    features: ['input', 'question', 'context', 'answer', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'errors_to_correct', 'corrected_responses', 'verified_response_mask'],
    num_rows: 70203
})


In [26]:
samples_to_include = 400
for i in range(samples_to_include):
    final_context_dataset.extend(format_single_correct_sample(correct_samples[i]))

random.shuffle(final_context_dataset)
print(len(final_context_dataset))

608


In [27]:
output_path = os.path.join(project_root, "context_qa_train_dataset_s2.json")
with open(output_path, "w") as f:
    json.dump(final_context_dataset, f, indent=4)

### Dataset fusion

In [28]:
# load 2 datasets and fuse them
math_dataset = datasets.load_dataset("json", data_files=f"{project_root}/math_qa_train_dataset_s2.json")
context_dataset = datasets.load_dataset("json", data_files=f"{project_root}/context_qa_train_dataset_s2.json")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [29]:
final_dataset = []
math_dataset = math_dataset["train"].to_list()
context_dataset = context_dataset["train"].to_list()

final_dataset.extend(math_dataset)
final_dataset.extend(context_dataset)
random.shuffle(final_dataset)
print(len(final_dataset))

1339


In [30]:
output_path = os.path.join(project_root, "final_train_dataset_s2.json")
with open(output_path, "w") as f:
    json.dump(final_dataset, f, indent=4)

In [1]:
import datasets
from datasets import Dataset, DatasetDict
from huggingface_hub import HfFolder

import sys
import os
# Add the project root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
data_path = "../../dataset/final_train_dataset.json"
dataset = datasets.load_dataset("json", data_files=data_path)
dataset = dataset["train"].to_list()

In [3]:
len(dataset)

35022

In [ ]:
HUGGINGFACE_TOKEN_ENV_VAR = ""
repo_name = "MathBite/llama_sa_self-correction"

In [5]:
token = HUGGINGFACE_TOKEN_ENV_VAR
if not token:
    raise ValueError(f"Hugging Face token not found. Set the {HUGGINGFACE_TOKEN_ENV_VAR} environment variable.")

HfFolder.save_token(token)

hf_dataset = Dataset.from_list(dataset)
dataset_dict = DatasetDict({"train": hf_dataset})

dataset_dict.push_to_hub(repo_name, private=True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/36 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/773 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/MathBite/llama_sa_self-correction/commit/b4af5888a179aa71dc8fc1433dd86e1f2739e092', commit_message='Upload dataset', commit_description='', oid='b4af5888a179aa71dc8fc1433dd86e1f2739e092', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/MathBite/llama_sa_self-correction', endpoint='https://huggingface.co', repo_type='dataset', repo_id='MathBite/llama_sa_self-correction'), pr_revision=None, pr_num=None)